<a href="https://colab.research.google.com/github/Muskan-Sh-23/Machine-learning/blob/main/NaiveBayes/Naive_Bayes_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_iris

# Load the dataset
iris = load_iris()
X = iris.data
y = iris.target

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Feature names:", iris.feature_names)
print("Species:", iris.target_names)


Dataset shape: (150, 4)
Target shape: (150,)
Feature names: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Species: ['setosa' 'versicolor' 'virginica']


In [2]:
# Shuffle indices reproducibly
np.random.seed(42)
indices = np.random.permutation(len(X))

# 80% training, 20% testing
test_size = int(0.2 * len(X))

test_indices = indices[:test_size]
train_indices = indices[test_size:]

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


Training samples: 120
Testing samples: 30


In [3]:

class GaussianNaiveBayesScratch:

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {}
        self.means = {}
        self.variances = {}

        for c in self.classes:
            # Samples belonging to this class
            X_c = X[y == c]

            # Prior probability of the class
            self.priors[c] = len(X_c) / len(X)

            # Mean and variance of each feature
            self.means[c] = np.mean(X_c, axis=0)

            # Small epsilon prevents zero variance
            self.variances[c] = (
                np.var(X_c, axis=0) + 1e-9
            )

    def calculate_log_likelihood(self, x, c):
        mean = self.means[c]
        variance = self.variances[c]

        # Log of Gaussian probability density
        log_likelihood = -0.5 * np.sum(
            np.log(2 * np.pi * variance)
            + ((x - mean) ** 2) / variance
        )

        return log_likelihood

    def predict_one(self, x):
        scores = {}

        for c in self.classes:
            log_prior = np.log(self.priors[c])

            log_likelihood = self.calculate_log_likelihood(
                x, c
            )

            # Log posterior, ignoring the common denominator
            scores[c] = log_prior + log_likelihood

        return max(scores, key=scores.get)

    def predict(self, X):
        predictions = []

        for x in X:
            predictions.append(self.predict_one(x))

        return np.array(predictions)


In [4]:

# Train our implementation
model = GaussianNaiveBayesScratch()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Calculate accuracy from scratch
correct = np.sum(y_pred == y_test)
accuracy = correct / len(y_test)

print("Correct predictions:", correct, "out of", len(y_test))
print("Test accuracy:", round(accuracy * 100, 2), "%")


Correct predictions: 30 out of 30
Test accuracy: 100.0 %


In [5]:

x = X_test[0]
actual_class = y_test[0]

print("Flower measurements:", x)
print("Actual species:", iris.target_names[actual_class])

for c in model.classes:
    log_prior = np.log(model.priors[c])
    log_likelihood = model.calculate_log_likelihood(x, c)
    score = log_prior + log_likelihood

    print(
        iris.target_names[c],
        "| Log prior:", round(log_prior, 3),
        "| Log likelihood:", round(log_likelihood, 3),
        "| Total score:", round(score, 3)
    )

print("Predicted species:", iris.target_names[model.predict_one(x)])


Flower measurements: [6.1 2.8 4.7 1.2]
Actual species: versicolor
setosa | Log prior: -1.099 | Log likelihood: -205.686 | Total score: -206.784
versicolor | Log prior: -1.074 | Log likelihood: -0.269 | Total score: -1.343
virginica | Log prior: -1.124 | Log likelihood: -5.649 | Total score: -6.773
Predicted species: versicolor
